# 01 — Clean seizure CSV and extract metadata

This notebook imports `../data/240619_Seizures_plotData.csv`, parses block-style metadata, removes summary rows, and exports clean event-level seizure data.

Expected input:
- Rows like `M15 pre CSD` or `M15 post CSD` define animal and stage.
- Rows like `Sham`, `sham`, or `CCI` define condition.
- Rows like `total seizures = ...`, `.../min`, `AVG`, and `SEM` are metadata/summary rows.
- Numeric rows are individual seizure/event measurements.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

data_path = Path("../data/240619_Seizures_plotData.csv")
output_dir = Path("../outputs/seizure_analysis")
output_dir.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(data_path)

print("Raw shape:", raw.shape)
display(raw.head(20))
display(raw.tail(10))

Raw shape: (673, 8)


,Unnamed: 0,amplitude,FWHM,prominence,AUC,rise,decay,duration
0,M15 pre CSD,0.189530797,11.92850904,0.131078746,3.413801929,27.79166667,15.20833333,43
1,Sham,0.30082629,21.60212123,0.158804082,6.627933861,11.64516129,17.35483871,29
2,NaN,0.302680952,33.07596846,0.141533188,6.985216869,10.86666667,18.13333333,29
3,NaN,0.371592378,23.43480083,0.155422561,11.69008973,16.70588235,23.29411765,40
4,total seizures = 32,0.315064949,33.65933818,0.138499456,6.304591116,12.7826087,10.2173913,23
5,NaN,0.337296866,67.07371877,0.155679518,6.692575354,11.13636364,10.86363636,22
6,3.48/min,0.392072743,79.62663722,0.143255025,11.95135478,16.025,17.975,34
7,NaN,0.362337647,88.32437392,0.183586804,11.26992345,23.44871795,11.55128205,35
8,NaN,0.299252887,72.90387218,0.132349996,6.030344907,9.695652174,15.30434783,25
9,NaN,0.473439306,15.43806631,0.12893423,11.67806721,20.28571429,9.714285714,30


,Unnamed: 0,amplitude,FWHM,prominence,AUC,rise,decay,duration
663,NaN,0.156893824,18.17322598,0.116503937,3.242049713,18.23026316,11.76973684,30
664,NaN,0.145957846,11.42338173,0.08831084,2.148397113,9.857142857,11.14285714,21
665,NaN,0.162366318,15.87435525,0.101107708,4.809778896,24.24056604,18.75943396,43
666,NaN,0.151890441,25.18012802,0.084585769,5.77812951,34.54385965,16.45614035,51
667,NaN,0.154752697,8.753580402,0.084508736,4.117777793,15.3015873,23.6984127,39
668,NaN,0.155280444,10.67063584,0.069283346,2.107660624,8.281690141,8.718309859,17
669,NaN,0.200549476,16.3546564,0.140858828,4.59175245,21.52531646,12.47468354,34
670,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
671,AVG,0.189168437,13.10623381,0.117611213,4.89649534,14.11544629,14.5697389,28.68518519
672,SEM,0.012412588,0.481077464,0.012734526,0.662248465,0.735399315,0.661257402,1.270634338


In [2]:
# Standardize column names

raw = raw.rename(columns={
    "Unnamed: 0": "label",
    "FWHM ": "FWHM"
})

feature_cols = [
    "amplitude",
    "FWHM",
    "prominence",
    "AUC",
    "rise",
    "decay",
    "duration"
]

for col in feature_cols:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

display(raw.dtypes)

label             str
amplitude     float64
FWHM          float64
prominence    float64
AUC           float64
rise          float64
decay         float64
duration      float64
dtype: object

In [3]:
# Parse block metadata and keep only numeric event rows

animal_pattern = re.compile(r"^(M\d+)\s+(pre|post)\s+CSD$", re.IGNORECASE)

current_animal = None
current_stage = None
current_condition = None
current_total_seizures = np.nan
current_rate_per_min = np.nan

event_rows = []
metadata_rows = []

for idx, row in raw.iterrows():

    label = row["label"]
    label_str = str(label).strip() if pd.notna(label) else ""

    animal_match = animal_pattern.match(label_str)

    if animal_match:
        current_animal = animal_match.group(1)
        current_stage = animal_match.group(2).lower() + "_CSD"
        current_condition = None
        current_total_seizures = np.nan
        current_rate_per_min = np.nan

        metadata_rows.append({
            "source_row": idx,
            "animal": current_animal,
            "stage": current_stage,
            "condition": current_condition,
            "metadata_type": "animal_stage",
            "metadata_value": label_str
        })
        continue

    if label_str.lower() in ["sham", "cci"]:
        current_condition = label_str.upper() if label_str.lower() == "cci" else "Sham"

        metadata_rows.append({
            "source_row": idx,
            "animal": current_animal,
            "stage": current_stage,
            "condition": current_condition,
            "metadata_type": "condition",
            "metadata_value": label_str
        })
        continue

    if label_str.lower().startswith("total seizures"):
        match = re.search(r"(\d+)", label_str)
        if match:
            current_total_seizures = int(match.group(1))

        metadata_rows.append({
            "source_row": idx,
            "animal": current_animal,
            "stage": current_stage,
            "condition": current_condition,
            "metadata_type": "total_seizures",
            "metadata_value": current_total_seizures
        })
        continue

    if "/min" in label_str.lower():
        match = re.search(r"([0-9]*\.?[0-9]+)", label_str)
        if match:
            current_rate_per_min = float(match.group(1))

        metadata_rows.append({
            "source_row": idx,
            "animal": current_animal,
            "stage": current_stage,
            "condition": current_condition,
            "metadata_type": "rate_per_min",
            "metadata_value": current_rate_per_min
        })
        continue

    if label_str.upper() in ["AVG", "SEM"]:
        metadata_rows.append({
            "source_row": idx,
            "animal": current_animal,
            "stage": current_stage,
            "condition": current_condition,
            "metadata_type": label_str.upper(),
            "metadata_value": label_str
        })
        continue

    if label_str and row[feature_cols].notna().sum() == 0:
        metadata_rows.append({
            "source_row": idx,
            "animal": current_animal,
            "stage": current_stage,
            "condition": current_condition,
            "metadata_type": "note",
            "metadata_value": label_str
        })
        continue

    if row[feature_cols].notna().sum() > 0 and current_animal is not None:

        event = {
            "source_row": idx,
            "event_id": len(event_rows),
            "animal": current_animal,
            "condition": current_condition if current_condition is not None else "Unknown",
            "stage": current_stage,
            "total_seizures_block": current_total_seizures,
            "rate_per_min_block": current_rate_per_min,
        }

        for col in feature_cols:
            event[col] = row[col]

        event_rows.append(event)

events = pd.DataFrame(event_rows)
block_metadata = pd.DataFrame(metadata_rows)

print("Clean event shape:", events.shape)
display(events.head())
display(events.tail())

print("Metadata shape:", block_metadata.shape)
display(block_metadata.head(20))

Clean event shape: (527, 14)


,source_row,event_id,animal,condition,stage,total_seizures_block,rate_per_min_block,amplitude,FWHM,prominence,AUC,rise,decay,duration
0,2,0,M15,Sham,pre_CSD,NaN,NaN,0.302681,33.075968,0.141533,6.985217,10.866667,18.133333,29.0
1,3,1,M15,Sham,pre_CSD,NaN,NaN,0.371592,23.434801,0.155423,11.690090,16.705882,23.294118,40.0
2,5,2,M15,Sham,pre_CSD,32.0,NaN,0.337297,67.073719,0.155680,6.692575,11.136364,10.863636,22.0
3,7,3,M15,Sham,pre_CSD,32.0,3.48,0.362338,88.324374,0.183587,11.269923,23.448718,11.551282,35.0
4,8,4,M15,Sham,pre_CSD,32.0,3.48,0.299253,72.903872,0.132350,6.030345,9.695652,15.304348,25.0


,source_row,event_id,animal,condition,stage,total_seizures_block,rate_per_min_block,amplitude,FWHM,prominence,AUC,rise,decay,duration
522,665,522,M18,Unknown,post_CSD,NaN,5.86,0.162366,15.874355,0.101108,4.809779,24.240566,18.759434,43.0
523,666,523,M18,Unknown,post_CSD,NaN,5.86,0.151890,25.180128,0.084586,5.778130,34.543860,16.456140,51.0
524,667,524,M18,Unknown,post_CSD,NaN,5.86,0.154753,8.753580,0.084509,4.117778,15.301587,23.698413,39.0
525,668,525,M18,Unknown,post_CSD,NaN,5.86,0.155280,10.670636,0.069283,2.107661,8.281690,8.718310,17.0
526,669,526,M18,Unknown,post_CSD,NaN,5.86,0.200549,16.354656,0.140859,4.591752,21.525316,12.474684,34.0


Metadata shape: (78, 6)


,source_row,animal,stage,condition,metadata_type,metadata_value
0,0,M15,pre_CSD,NaN,animal_stage,M15 pre CSD
1,1,M15,pre_CSD,Sham,condition,Sham
2,4,M15,pre_CSD,Sham,total_seizures,32
3,6,M15,pre_CSD,Sham,rate_per_min,3.48
4,33,M15,pre_CSD,Sham,AVG,AVG
5,34,M15,pre_CSD,Sham,SEM,SEM
6,39,M15,post_CSD,NaN,animal_stage,M15 post CSD
7,40,M15,post_CSD,Sham,condition,Sham
8,43,M15,post_CSD,Sham,total_seizures,52
9,46,M15,post_CSD,Sham,rate_per_min,4.08


In [4]:
print("Missing values per column:")
display(events.isna().sum())

print("Counts by condition/stage:")
display(pd.crosstab(events["condition"], events["stage"]))

print("Events per animal/stage:")
display(
    events
    .groupby(["animal", "condition", "stage"])
    .size()
    .reset_index(name="n_events")
)

Missing values per column:


source_row                0
event_id                  0
animal                    0
condition                 0
stage                     0
total_seizures_block    277
rate_per_min_block       47
amplitude                 0
FWHM                      0
prominence                0
AUC                       0
rise                      0
decay                     0
duration                  0
dtype: int64

Counts by condition/stage:


stage,post_CSD,pre_CSD
condition,,
CCI,177,96
Sham,73,57
Unknown,107,17


Events per animal/stage:


,animal,condition,stage,n_events
0,M13,CCI,post_CSD,89
1,M13,CCI,pre_CSD,40
2,M15,Sham,post_CSD,49
3,M15,Sham,pre_CSD,28
4,M16,CCI,post_CSD,62
5,M16,CCI,pre_CSD,30
6,M18,Unknown,post_CSD,107
7,M18,Unknown,pre_CSD,17
8,M19,CCI,post_CSD,26
9,M19,CCI,pre_CSD,26


In [5]:
events_path = output_dir / "seizure_events_clean.csv"
metadata_path = output_dir / "seizure_block_metadata.csv"

events.to_csv(events_path, index=False)
block_metadata.to_csv(metadata_path, index=False)

print("Saved:")
print(events_path)
print(metadata_path)

Saved:
../outputs/seizure_analysis/seizure_events_clean.csv
../outputs/seizure_analysis/seizure_block_metadata.csv
